# Lab: GenAI, LLMs and APIs — OpenAI GPT API Exercises

**Goal.** Build small, reusable applications for conversation, summarization, translation, sentiment analysis, and text completion; understand generation parameters; and compare the OpenAI and Google Vertex AI integration patterns.

This notebook is designed to run **from top to bottom without API credentials**. It first demonstrates the tasks with transparent local examples, then provides optional live API paths. Local examples are instructional demonstrations, not substitutes for a real language model. Live API calls may incur charges.

**Learning outcomes**
- Configure API clients safely with environment variables.
- Use structured prompts and validate outputs.
- Understand which generation parameters are supported by current APIs and which belong to older completion APIs.
- Test outputs and discuss limitations, privacy, and cost.

## 0. Setup and safe configuration

Install dependencies if needed:

```bash
pip install openai google-genai python-dotenv pandas
```

Set `OPENAI_API_KEY` and optionally `OPENAI_MODEL` in your environment (or a local `.env` file). Never paste keys into a notebook, commit them to Git, or include personal/sensitive data in prompts without organisational approval. The live API calls below are opt-in and remain disabled unless a key is present.

In [1]:
!pip install openai google-genai python-dotenv pandas

In [2]:
import os
import json
import re
from typing import Literal

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-nano")
USE_LIVE_OPENAI = bool(OPENAI_API_KEY)

print("Live OpenAI calls enabled:", USE_LIVE_OPENAI)
print("Model configured:", OPENAI_MODEL if USE_LIVE_OPENAI else "offline demonstration mode")

# Do not print the API key. If it is missing, the notebook remains runnable offline.

Live OpenAI calls enabled: True
Model configured: gpt-5.4-nano


In [3]:
%pip install tiktoken

import os
import json
import re
import time

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal
import tiktoken

pd.set_option("display.max_colwidth", 120)

load_dotenv()                                   # reads OPENAI_API_KEY from the .env file
print("Key found:", "OPENAI_API_KEY" in os.environ)   # never print the key itself

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
MODEL = "gpt-5.4-nano"                          # the cheapest current model; change it in one place

if client is None:
	print("OpenAI client not created; live API cells will be skipped.")

Note: you may need to restart the kernel to use updated packages.
Key found: True


In [4]:
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-nano")
USE_LIVE_OPENAI = bool(OPENAI_API_KEY)

print("Live OpenAI calls enabled:", USE_LIVE_OPENAI)
print("Model configured:", OPENAI_MODEL if USE_LIVE_OPENAI else "offline demonstration mode")

# Do not print the API key. If it is missing, the notebook remains runnable offline.

Live OpenAI calls enabled: True
Model configured: gpt-5.4-nano


In [5]:
if client is None:
	print("Model listing skipped: configure OPENAI_API_KEY to enable live OpenAI calls.")
else:
	available = sorted(m.id for m in client.models.list())
	print(len(available), "models, for example:", [m for m in available if m.startswith("gpt-5.4")])

142 models, for example: ['gpt-5.4', 'gpt-5.4-2026-03-05', 'gpt-5.4-mini', 'gpt-5.4-mini-2026-03-17', 'gpt-5.4-nano', 'gpt-5.4-nano-2026-03-17', 'gpt-5.4-pro', 'gpt-5.4-pro-2026-03-05']


In [6]:
if client is None:
    print("[Offline demo] A large language model is an AI system that can understand and generate text.")
else:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "In one sentence, what is a large language model?"}],
    )
    print(response.choices[0].message.content)

A large language model is an AI system trained on massive amounts of text to learn patterns of language and generate or understand new text in response to prompts.


### API parameter guide: modern APIs vs. legacy exercises

The original exercise lists parameters from older text-completion endpoints. Current chat/responses APIs do **not** support every legacy parameter, and parameter availability varies by model and endpoint.

| Parameter | General effect | Practical caution |
|---|---|---|
| `temperature` | Higher values generally increase variation; lower values make output more focused | Some models/endpoints restrict sampling controls |
| `top_p` | Nucleus sampling; limits choices to a probability mass | Usually adjust either `temperature` or `top_p`, not both |
| `max_tokens` / `max_output_tokens` | Limits generated output | Exact name depends on endpoint; truncation is possible |
| `frequency_penalty` | Discourages repeated tokens | May reduce natural repetition when repetition is appropriate |
| `presence_penalty` | Encourages introducing tokens/topics not already used | Can cause unnecessary topic drift |
| `n` | Requests multiple candidates on supported endpoints | Increases cost; not supported everywhere |
| `stop` | Stops generation at specified sequences on supported endpoints | May cut off a useful answer; availability varies |
| `best_of` | Legacy Completions API parameter that samples candidates and returns the best | Not a general current Chat Completions/Responses parameter |
| `echo` | Legacy Completions API option to include the prompt | Not a general current chat parameter |
| `logprobs` | Requests token log-probabilities on supported endpoints | Availability and response shape depend on model/endpoint |
| `logit_bias` | Adjusts likelihood of specific token IDs | Requires token IDs; can make output awkward or unreliable |

**Key lesson:** do not blindly copy parameters from an old tutorial. Check the documentation for the exact endpoint and model you are using.

## Shared helper: live API or offline demonstration

The offline branch is intentionally simple and labelled as a demonstration. It allows all exercises and checks to run without network access or credentials. When a key is configured, the helper calls the current OpenAI Responses API.

In [7]:
def ask_model(instructions: str, input_text: str, *, temperature: float = 0.2,
              max_output_tokens: int = 250) -> str:
    """Call OpenAI when configured; otherwise return a clearly labelled local demo."""
    if USE_LIVE_OPENAI:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        response = client.responses.create(
            model=OPENAI_MODEL,
            instructions=instructions,
            input=input_text,
            temperature=temperature,
            max_output_tokens=max_output_tokens,
        )
        return response.output_text.strip()
    return offline_demo(instructions, input_text)


def offline_demo(instructions: str, input_text: str) -> str:
    """Small deterministic stand-in used only to exercise notebook flow offline."""
    task = instructions.lower()
    text = input_text.strip()
    if "sentiment" in task:
        low = text.lower()
        positive = any(w in low for w in ["love", "great", "excellent", "happy", "wonderful", "good"])
        negative = any(w in low for w in ["hate", "terrible", "awful", "sad", "bad", "disappointed"])
        label = "positive" if positive and not negative else "negative" if negative and not positive else "neutral"
        return json.dumps({"sentiment": label, "confidence": "heuristic demo; not a calibrated probability"})
    if "translate" in task:
        return "[Offline demo] Translation requires a language model; input preserved: " + text
    if "summar" in task:
        sentences = re.split(r"(?<=[.!?])\s+", text)
        return " ".join(sentences[:2])[:500]
    if "complete" in task or "continue" in task:
        return text + " [Offline demo: a live model would continue this text.]"
    return "[Offline demo] " + text[:400]

print(ask_model("Answer briefly about technology.", "What is machine learning?"))

Machine learning is a way to build software that learns patterns from data instead of being explicitly programmed with fixed rules. A model is trained on examples (data), finds relationships, and then uses what it learned to make predictions or decisions on new, unseen inputs.


# LAB GenAI - LLMs - OpenAI GPT API Exercises

## 1. Basic Conversation
**Exercise:** Create a simple chatbot that can answer basic questions about a given topic (e.g., history, technology).  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `stop`.

Comment what happen when you change the parameters 
(read documentation!)

In [8]:
# Basic conversation
# Use a clear system instruction and a focused question. 
# Compare a low temperature (more focused) with a higher 
# temperature (often more varied) in live mode. Temperature does not guarantee factuality.

conversation_instruction = (
    "You are a friendly science tutor. Answer accurately in plain language, "
    "define technical terms, and say when you are uncertain. Keep the answer under 120 words."
)
question = "Explain how a fuel cell converts chemical energy into electricity."
conversation_answer = ask_model(conversation_instruction, question, temperature=0.2, max_output_tokens=180)
print(conversation_answer)

# Experiment: try temperature=0.8 and compare the wording. The answer may vary between runs.

A fuel cell converts chemical energy into electricity through an electrochemical reaction, not burning fuel.

- **Fuel (like hydrogen)** enters the **anode** and is split into **electrons** and **ions**.  
- The **electrons** can’t pass through the electrolyte, so they travel through an **external circuit**, creating an electric current.  
- **Ions** move through the **electrolyte** to the **cathode**.  
- At the **cathode**, the ions combine with **oxygen** (from air) and **electrons** to form **water** (or other products).  

Overall, the reactants’ chemical energy becomes electrical energy plus heat.


In [9]:
conversation_instruction = (
    "You are a friendly science tutor. Answer accurately in plain language, "
    "define technical terms, and say when you are uncertain. Keep the answer under 120 words."
)
question = "Explain how a fuel cell converts chemical energy into electricity."
conversation_answer = ask_model(conversation_instruction, question, temperature=0.8, max_output_tokens=180)
print(conversation_answer)

A fuel cell converts **chemical energy** directly into **electrical energy** through **electrochemical reactions** (not burning).

- **Fuel** (like hydrogen) enters the **anode**. It breaks into **electrons** and **ions**.  
- The **electrons** flow through an external circuit, creating **electric current**.  
- **Ions** move through an **electrolyte** to the **cathode**.  
- At the **cathode**, **oxygen** (from air) combines with the ions and electrons to form **water** (or other products).  

Overall, the reactions transfer energy to electrons, which you capture as electricity.


### Reflection — conversation parameters

- A lower temperature is useful for concise, consistent explanations; higher values can help brainstorming but may introduce unsupported details.
- `max_output_tokens` limits length, not factual quality.
- `top_p`, penalties, `n`, and `stop` should only be passed if the chosen model/endpoint supports them. The legacy `best_of` option is not interchangeable with `n`.

## 2. Summarization
**Exercise:** Write a script that takes a long text input and summarizes it into a few sentences.  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `best_of`, `logprobs`.

Comment what happen when you change the parameters 
(read documentation!)

In [10]:
# Summarization

# A good summary preserves the main claims and avoids adding information. 
# We ask for a short, neutral summary and use a deliberately long sample text.

article = """
Hydrogen fuel cells generate electricity through electrochemical reactions rather than combustion.
In a proton-exchange membrane fuel cell, hydrogen is oxidised at the anode, producing protons and electrons.
The membrane transports protons while electrons travel through an external circuit, delivering useful electrical work.
At the cathode, oxygen reacts with protons and electrons to form water. Heat and water are important outputs.
Performance depends on catalyst activity, membrane hydration, gas transport, operating temperature, and degradation.
Platinum is commonly used as a catalyst, but cost and durability remain important engineering challenges.
Researchers investigate alloy catalysts, nanostructures, and improved catalyst-layer architectures to reduce platinum loading
while maintaining activity and long-term stability. System-level efficiency also depends on how hydrogen is produced,
stored, and delivered; a fuel cell itself does not determine whether the overall energy pathway is low-carbon.
""".strip()
summary = ask_model(
    "Summarize the text in 2-3 sentences. Preserve key facts, do not add claims, and use neutral language.",
    article, temperature=0.1, max_output_tokens=120
)
print(summary)

Hydrogen fuel cells produce electricity via electrochemical reactions: in a proton-exchange membrane fuel cell, hydrogen is oxidized at the anode to form protons and electrons, protons pass through the membrane while electrons flow through an external circuit, and oxygen at the cathode combines with protons and electrons to form water. Performance depends on factors such as catalyst activity, membrane hydration, gas transport, operating temperature, and degradation, with platinum commonly used as a catalyst despite cost and durability challenges. Research focuses on reducing platinum loading using alloy catalysts, nanostructures, and improved catalyst


### Reflection — summarization parameters

- Lower temperature often supports stable, faithful summaries, but outputs still need checking against the source.
- Increase output-token limits if key points are omitted; an overly small limit can truncate the summary.
- `best_of` is a legacy completion parameter, not a generic modern API setting. `logprobs` is only available for supported models/endpoints and is not itself a summary-quality score.

## 3. Translation
**Exercise:** Develop a tool that translates text from one language to another using the API.  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `echo`, `logit_bias`.

Comment what happen when you change the parameters 
(read documentation!)

In [11]:
## Translation

# Specify both source and target languages, preserve meaning, and avoid 
# adding explanations unless requested. Translation quality should be evaluated 
# for meaning, register, terminology, and cultural nuance.

source_text = "The catalyst layer must balance high electrochemical activity with long-term stability."
translation = ask_model(
    "Translate the input from English to German. Return only the translation; preserve scientific terminology.",
    source_text, temperature=0.1, max_output_tokens=100
)
print("English:", source_text)
print("German:", translation)

English: The catalyst layer must balance high electrochemical activity with long-term stability.
German: Die Katalysatorschicht muss eine hohe elektrochemische Aktivität mit einer langfristigen Stabilität in Einklang bringen.


### Reflection — translation parameters

- Lower temperature can improve consistency for technical translation, but it does not guarantee correctness.
- `echo` belongs to older completion APIs and is not a standard chat translation option.
- `logit_bias` requires token IDs and is rarely a good way to control translation terminology; provide a glossary in the prompt instead.

## 4. Sentiment Analysis
**Exercise:** Implement a sentiment analysis tool that determines the sentiment of a given text (positive, negative, neutral).  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `logprobs`.

Comment what happen when you change the parameters 
(read documentation!)

In [12]:
## Sentiment analysis

# For machine-readable output, ask for a strict schema 
# and validate it. Sentiment labels are context-sensitive: sarcasm, 
# mixed feelings, negation, and domain-specific wording can confuse simple approaches.

sentiment_examples = [
    "The support team solved my issue quickly. Excellent service!",
    "I am disappointed; the product stopped working after one day.",
    "The package arrived on Tuesday.",
]

def analyze_sentiment(text: str) -> dict:
    raw = ask_model(
        'Classify sentiment as positive, negative, or neutral. Return JSON only with keys '
        '"sentiment" and "rationale". Do not infer sentiment without evidence.',
        text, temperature=0, max_output_tokens=100
    )
    try:
        result = json.loads(raw)
        label = result.get("sentiment")
        if label not in {"positive", "negative", "neutral"}:
            raise ValueError("Unexpected sentiment label")
        return {"sentiment": label, "rationale": str(result.get("rationale", result.get("confidence", "")))}
    except (json.JSONDecodeError, ValueError):
        # Make malformed output visible instead of silently pretending it is valid.
        return {"sentiment": "unparsed", "rationale": raw}

for example in sentiment_examples:
    print(example, "=>", analyze_sentiment(example))

The support team solved my issue quickly. Excellent service! => {'sentiment': 'positive', 'rationale': "The user reports that the support team solved the issue quickly and praises the service as 'Excellent,' indicating clear positive sentiment."}
I am disappointed; the product stopped working after one day. => {'sentiment': 'negative', 'rationale': 'The user expresses disappointment and reports the product stopped working after one day, indicating dissatisfaction.'}
The package arrived on Tuesday. => {'sentiment': 'neutral', 'rationale': 'The statement is a factual timeline update about the package arriving on Tuesday and does not express approval, disappointment, or any emotional tone.'}


### Reflection — sentiment parameters

- Temperature near zero tends to improve label consistency, but outputs still require evaluation against labelled examples.
- Multiple samples (`n`, where supported) can reveal uncertainty but add cost and do not automatically improve accuracy.
- `logprobs` may help inspect token-level model confidence when supported; token likelihood is not the same as calibrated confidence in the sentiment label.

## 5. Text Completion
**Exercise:** Create a text completion application that generates text based on an initial prompt.  
**Parameters to explore:** `temperature`, `max_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `stop`, `best_of`.

Comment what happen when you change the parameters 
(read documentation!)

In [13]:
# Text completion

# Continue a supplied opening while following constraints. Use a clear instruction to avoid unbounded or irrelevant continuation.

opening = "A promising catalyst-layer design should improve fuel-cell performance by"
completion = ask_model(
    "Continue the sentence in 2-4 technically grounded sentences. Avoid invented numerical results.",
    opening, temperature=0.5, max_output_tokens=140
)
print(completion)

A promising catalyst-layer design should improve fuel-cell performance by enhancing reactant transport (reducing mass-transport losses) while maintaining efficient electron pathways through a well-connected conductive network. It should also increase electrochemically active surface area and optimize catalyst utilization by tailoring pore structure and ionomer distribution for effective proton conduction. Finally, it should mitigate degradation by improving tolerance to water management and limiting catalyst agglomeration and corrosion under operating conditions.


### Reflection — text-completion parameters

- Higher temperature can encourage creative alternatives; lower temperature is often preferable for technical writing.
- A `stop` sequence can end generation at a chosen marker on supported endpoints, but may cut off a sentence.
- `best_of` is associated with legacy completion workflows and should not be assumed available in current APIs.

## Lightweight evaluation and testing

Even a small lab should test expected behavior. These checks verify local plumbing and output structure, not real-model quality.

In [14]:
# Unit checks for the offline helper and sentiment output shape.
assert isinstance(conversation_answer, str) and len(conversation_answer) > 0
assert isinstance(summary, str) and len(summary) > 0
assert isinstance(translation, str) and len(translation) > 0
assert all(analyze_sentiment(x)["sentiment"] in {"positive", "negative", "neutral", "unparsed"}
           for x in sentiment_examples)
assert isinstance(completion, str) and len(completion) > 0
print("All notebook plumbing checks passed.")

All notebook plumbing checks passed.


# BONUS: Google Vertex AI

## 1. Basic Conversation
**Exercise:** Create a basic chatbot using Google Vertex AI to answer questions about a given topic.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `stop`.

Comment what happen when you change the parameters 
(read documentation!)

## Bonus: Google Vertex AI / Gemini

The bonus section mirrors the five tasks using Google's current `google-genai` SDK. It is optional and will run only if Google credentials are configured. For Vertex AI, configure the project and region as described in Google's current documentation; for example, set `GOOGLE_CLOUD_PROJECT` and `GOOGLE_CLOUD_LOCATION`, and authenticate with Application Default Credentials. Do not put service-account keys in the notebook.

The exact supported generation settings vary by model. Vertex/Gemini parameter names and behavior differ from OpenAI's; do not assume `frequency_penalty`, `presence_penalty`, `best_of`, `echo`, or `logit_bias` map directly.

In [17]:
!pip install google-genai

In [18]:
GOOGLE_CLOUD_PROJECT = os.getenv("GOOGLE_CLOUD_PROJECT")
GOOGLE_CLOUD_LOCATION = os.getenv("GOOGLE_CLOUD_LOCATION", "us-central1")
USE_VERTEX = bool(GOOGLE_CLOUD_PROJECT)


def ask_vertex(prompt: str, *, temperature: float = 0.2, max_output_tokens: int = 250) -> str:
    """Optional Gemini call through Vertex AI; raises a clear setup error if not configured."""
    if not USE_VERTEX:
        return "[Vertex AI skipped] Set GOOGLE_CLOUD_PROJECT, configure ADC, and install google-genai to run live."
    try:
        from google import genai
        from google.genai import types
    except ImportError as exc:
        raise ImportError("Install the SDK with: pip install google-genai") from exc
    client = genai.Client(vertexai=True, project=GOOGLE_CLOUD_PROJECT, location=GOOGLE_CLOUD_LOCATION)
    response = client.models.generate_content(
        model=os.getenv("VERTEX_MODEL", "gemini-2.5-flash"),
        contents=prompt,
        config=types.GenerateContentConfig(temperature=temperature, max_output_tokens=max_output_tokens),
    )
    return (response.text or "").strip()

print("Vertex AI configured:", USE_VERTEX)
print(ask_vertex("In one sentence, explain what a language model does."))

Vertex AI configured: False
[Vertex AI skipped] Set GOOGLE_CLOUD_PROJECT, configure ADC, and install google-genai to run live.


In [19]:
# 1. Conversation
vertex_conversation = ask_vertex("You are a science tutor. Explain machine learning in two short sentences.")
print(vertex_conversation)


[Vertex AI skipped] Set GOOGLE_CLOUD_PROJECT, configure ADC, and install google-genai to run live.


## 2. Summarization
**Exercise:** Develop a script that summarizes long text inputs using Google Vertex AI.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `best_of`, `logprobs`.

Comment what happen when you change the parameters 
(read documentation!)

In [20]:
# 2. Summarization
vertex_summary = ask_vertex("Summarize in two sentences: " + article, temperature=0.1, max_output_tokens=120)
print(vertex_summary)


[Vertex AI skipped] Set GOOGLE_CLOUD_PROJECT, configure ADC, and install google-genai to run live.


## 3. Translation
**Exercise:** Create a tool that translates text from one language to another using Google Vertex AI.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `echo`, `logit_bias`.

Comment what happen when you change the parameters 
(read documentation!)

In [22]:
# 3. Translation
vertex_translation = ask_vertex("Translate into German, output only the translation: " + source_text,
                                temperature=0.1, max_output_tokens=100)
print(vertex_translation)

[Vertex AI skipped] Set GOOGLE_CLOUD_PROJECT, configure ADC, and install google-genai to run live.


## 4. Sentiment Analysis
**Exercise:** Implement a sentiment analysis tool using Google Vertex AI to determine the sentiment of a given text.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `n`, `logprobs`.

Comment what happen when you change the parameters 
(read documentation!)

In [23]:
# 4. Sentiment analysis
vertex_sentiment = ask_vertex('Return JSON with sentiment (positive, negative, neutral) and rationale: '
                              + sentiment_examples[0], temperature=0, max_output_tokens=100)
print(vertex_sentiment)

[Vertex AI skipped] Set GOOGLE_CLOUD_PROJECT, configure ADC, and install google-genai to run live.


## 5. Text Completion
**Exercise:** Develop a text completion application using Google Vertex AI to generate text based on an initial prompt.  
**Parameters to explore:** `temperature`, `max_output_tokens`, `top_p`, `frequency_penalty`, `presence_penalty`, `stop`, `best_of`.

Comment what happen when you change the parameters 
(read documentation!)

In [24]:
# 5. Text completion
vertex_completion = ask_vertex("Continue in two sentences without inventing measurements: " + opening,
                               temperature=0.5, max_output_tokens=140)
print(vertex_completion)

[Vertex AI skipped] Set GOOGLE_CLOUD_PROJECT, configure ADC, and install google-genai to run live.


### Vertex AI task examples

Each cell above is independently runnable. Without Vertex credentials it prints a setup message. With credentials it calls the configured Gemini model.

## Final conclusions

1. **Prompt design matters:** specify role, task, constraints, audience, and output format.
2. **Parameters are not universal:** use only parameters supported by the selected API endpoint and model; many names in the original exercise come from legacy text-completion APIs.
3. **Validate outputs:** parse structured output, check labels and required fields, and test edge cases.
4. **Evaluate quality separately from successful execution:** a cell running without errors does not prove that a summary is faithful or a translation accurate. Use representative labelled examples and human review.
5. **Protect data and credentials:** minimise sensitive input, use approved providers, keep keys in environment variables, and monitor usage/cost.
6. **Offline results are demonstrations only:** report real model metrics only after running with valid credentials and recording actual outputs.